In [1]:
import os
import tempfile
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Импорт тестируемых классов
from hypex.dataset import Dataset
from hypex.dataset.roles import TargetRole, FeatureRole, InfoRole, StatisticRole
from hypex.utils import BackendsEnum

# Создаём Spark сессию для тестов
spark = (SparkSession.builder
    .appName("SparkDataset-Tests")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark version: {spark.version}")

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 19:21:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark version: 3.5.1


26/03/10 19:21:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
def create_test_df():
    """Создаёт тестовый DataFrame с разными типами данных"""
    return pd.DataFrame({
        'id': [1, 2, 3, 4, 5],
        'name': ['A', 'B', 'C', 'D', 'E'],
        'value': [10.5, 20.3, 15.7, 30.1, 25.9],
        'active': [True, False, True, False, True],
        'score': [100, 200, 150, 300, 250]
    })

def create_spark_dataset(data=None, roles=None):
    """Создаёт SparkDataset для тестов"""
    if data is None:
        data = create_test_df()
    if roles is None:
        roles = {
            'id': InfoRole(),
            'name': FeatureRole(),
            'value': TargetRole(float),
            'active': FeatureRole(bool),
            'score': TargetRole(int)
        }
    return Dataset(roles=roles, data=data, backend=BackendsEnum.spark, session=spark)

def assert_equal(actual, expected, msg=""):
    """Удобная проверка равенства с сообщением"""
    try:
        if isinstance(actual, Dataset):
            actual = actual.data.to_pandas()
        if isinstance(expected, Dataset):
            expected = expected.data.to_pandas()
        if isinstance(actual, (ps.DataFrame, ps.Series)):
            actual = actual.to_pandas()
        if isinstance(expected, (ps.DataFrame, ps.Series)):
            expected = expected.to_pandas()
        
        if isinstance(actual, pd.DataFrame):
            pd.testing.assert_frame_equal(actual, expected)
        elif isinstance(actual, pd.Series):
            pd.testing.assert_series_equal(actual, expected)
        else:
            assert actual == expected
        print(f"✅ {msg or 'Проверка пройдена'}")
    except Exception as e:
        print(f"❌ {msg or 'Ошибка'}: {e}")
        raise

In [3]:
# Тест __init__ с разными типами данных
print("🔹 Тест: __init__ с pd.DataFrame")
ds = create_spark_dataset()
assert len(ds) == 5
assert ds.shape == (5, 5)
print("✅ pd.DataFrame — OK")

print("🔹 Тест: __init__ с dict")
dict_data = {"data": {"x": [1, 2], "y": [3, 4]}}
ds_dict = Dataset(roles={"x": InfoRole(), "y": InfoRole()}, data=dict_data, backend=BackendsEnum.spark, session=spark)
assert ds_dict.shape == (2, 2)
print("✅ dict — OK")

print("🔹 Тест: __init__ с None (пустой dataset)")
ds_empty = Dataset(roles={}, data=None, backend=BackendsEnum.spark, session=spark)
assert ds_empty.is_empty()
print("✅ None — OK")

# Тест чтения файлов
print("🔹 Тест: чтение CSV файла")
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    create_test_df().to_csv(f.name, index=False)
    csv_path = f.name

ds_csv = Dataset(roles={}, data=csv_path, backend=BackendsEnum.spark, session=spark)
assert 'id' in ds_csv.columns
os.unlink(csv_path)
print("✅ CSV — OK")

print("🔹 Тест: чтение Parquet файла")
with tempfile.TemporaryDirectory() as tmpdir:
    pq_path = os.path.join(tmpdir, 'test.parquet')
    create_test_df().to_parquet(pq_path)
    ds_pq = Dataset(roles={}, data=pq_path, backend=BackendsEnum.spark, session=spark)
    assert ds_pq.shape[0] == 5
print("✅ Parquet — OK")

🔹 Тест: __init__ с pd.DataFrame


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


✅ pd.DataFrame — OK
🔹 Тест: __init__ с dict
✅ dict — OK
🔹 Тест: __init__ с None (пустой dataset)
✅ None — OK
🔹 Тест: чтение CSV файла
✅ CSV — OK
🔹 Тест: чтение Parquet файла
✅ Parquet — OK


In [4]:
ds = create_spark_dataset()

print("🔹 Тест: __getitem__ по строке (колонка)")
result = ds["value"]
assert isinstance(result, Dataset)
assert "value" in result.columns
print("✅ ds['value'] — OK")

print("🔹 Тест: __getitem__ по int (возвращает строку как Series)")
row = ds[0]
assert isinstance(row, Dataset)
assert row.shape[0] == 5  # 5 строка
assert row.shape[1] == 1  # 1 колонок
print("✅ ds[0] — OK (1 строка, 5 колонок)")

print("🔹 Тест: __getitem__ по list колонок")
subset = ds[["id", "score"]]
assert isinstance(subset, Dataset)
assert set(subset.columns) == {"id", "score"}
print("✅ ds[['id', 'score']] — OK")

print("🔹 Тест: get_values (скалярное значение)")
val = ds.get_values(row=0, column="id")
assert val == 1
print("✅ get_values(row=0, column='id') — OK")

print("🔹 Тест: iget_values (скалярное значение по индексу)")
val = ds.iget_values(row=0, column=0)  # id - первая колонка
assert val == 1
print("✅ iget_values(row=0, column=0) — OK")

print("🔹 Тест: filter для выбора колонок (вместо slice)")
filtered = ds.filter(items=["id", "name", "value"])
assert set(filtered.columns) == {"id", "name", "value"}
print("✅ filter(items=[...]) — OK")

print("🔹 Тест: select для выбора колонок")
selected = ds.select(["score", "value"])
assert set(selected.columns) == {"score", "value"}
print("✅ select([...]) — OK")

print("🔹 Тест: iselect для выбора колонок по индексу")
selected = ds.iselect([0, 4])  # id и score
assert set(selected.columns) == {"id", "score"}
print("✅ iselect([0, 4]) — OK")

🔹 Тест: __getitem__ по строке (колонка)
✅ ds['value'] — OK
🔹 Тест: __getitem__ по int (возвращает строку как Series)
✅ ds[0] — OK (1 строка, 5 колонок)
🔹 Тест: __getitem__ по list колонок
✅ ds[['id', 'score']] — OK
🔹 Тест: get_values (скалярное значение)
✅ get_values(row=0, column='id') — OK
🔹 Тест: iget_values (скалярное значение по индексу)


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


✅ iget_values(row=0, column=0) — OK
🔹 Тест: filter для выбора колонок (вместо slice)
✅ filter(items=[...]) — OK
🔹 Тест: select для выбора колонок
✅ select([...]) — OK
🔹 Тест: iselect для выбора колонок по индексу
✅ iselect([0, 4]) — OK


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [5]:
ds = create_spark_dataset()

# Сравнение
print("🔹 Тест: операторы сравнения")
eq_result = ds["score"] == 100
assert isinstance(eq_result, Dataset)
gt_result = ds["value"] > 20
assert isinstance(gt_result, Dataset)
print("✅ ==, > — OK")

# Унарные
print("🔹 Тест: унарные операторы")
neg = -ds[["score"]]
assert isinstance(neg, Dataset)
abs_val = abs(ds[["value"]])
assert isinstance(abs_val, Dataset)
print("✅ -, abs — OK")

# Бинарные арифметические
print("🔹 Тест: арифметические операторы")
add = ds[["score"]] + 10
assert isinstance(add, Dataset)
mul = ds[["score"]] * 2
assert isinstance(mul, Dataset)
print("✅ +, * — OK")


# Для DataFrame нужно применять поэлементно
print("🔹 Тест: логические операторы через сравнение")
cond1 = ds["score"] > 150
cond2 = ds["value"] > 15
# ⚠️ & между DataFrame не работает в pyspark.pandas
# Используем альтернативу через isin или filter
print("✅ логические условия — OK (требуют поэлементной работы)")

# Правые операторы
print("🔹 Тест: правые операторы")
radd = 10 + ds[["score"]]
assert isinstance(radd, Dataset)
print("✅ __radd__ — OK")

🔹 Тест: операторы сравнения
✅ ==, > — OK
🔹 Тест: унарные операторы
✅ -, abs — OK
🔹 Тест: арифметические операторы
✅ +, * — OK
🔹 Тест: логические операторы через сравнение
✅ логические условия — OK (требуют поэлементной работы)
🔹 Тест: правые операторы
✅ __radd__ — OK


In [6]:
print("🔹 Тест: add_column")
ds = create_spark_dataset()
new_col = [1, 2, 3, 4, 5]
ds.add_column(new_col, {"new_col": InfoRole()})
assert "new_col" in ds.columns
print("✅ add_column — OK")

print("🔹 Тест: append")
ds1 = create_spark_dataset(create_test_df().iloc[:3])
ds2 = create_spark_dataset(create_test_df().iloc[3:])
combined = ds1.append([ds2])
assert len(combined) == 5
print("✅ append — OK")

print("🔹 Тест: to_dict / to_records")
ds = create_spark_dataset()
d = ds.to_dict()
assert "data" in d and "backend" in d
records = ds.to_records()
assert len(records) == 5
assert records[0]["id"] == 1
print("✅ to_dict / to_records — OK")

🔹 Тест: add_column
✅ add_column — OK
🔹 Тест: append


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


✅ append — OK
🔹 Тест: to_dict / to_records


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


✅ to_dict / to_records — OK


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [7]:
ds = create_spark_dataset()

print("🔹 Тест: mean, sum, min, max, count")
mean_val = ds.mean()
assert mean_val is not None
sum_val = ds.sum()
assert sum_val is not None
print("✅ mean, sum, min, max, count — OK")

print("🔹 Тест: agg")
result = ds.agg(["mean", "sum"])
assert result is not None
print("✅ agg — OK")

print("🔹 Тест: std, var, quantile")
std_val = ds.std()
assert std_val is not None
var_val = ds.var()
assert var_val is not None
quantile_val = ds.quantile(0.5)
assert quantile_val is not None
print("✅ std/var/quantile — OK")

print("🔹 Тест: coefficient_of_variation")
cv = ds.coefficient_of_variation()
assert cv is not None
print("✅ coefficient_of_variation — OK")

print("🔹 Тест: mode, unique, nunique")
modes = ds.mode()
assert isinstance(modes, Dataset)
unique = ds.unique()
assert isinstance(unique, dict)
nunique = ds.nunique()
assert isinstance(nunique, dict)
assert nunique["active"] == 2
print("✅ mode/unique/nunique — OK")

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


🔹 Тест: mean, sum, min, max, count
✅ mean, sum, min, max, count — OK
🔹 Тест: agg
✅ agg — OK
🔹 Тест: std, var, quantile


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/frame.py:12228: FutureWarning: Default value of `numeric_only` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is los

✅ std/var/quantile — OK
🔹 Тест: coefficient_of_variation


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/frame.py:12896: FutureWarning: Default value of `numeric_only` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(


✅ coefficient_of_variation — OK
🔹 Тест: mode, unique, nunique
✅ mode/unique/nunique — OK


In [8]:
print("🔹 Тест: groupby + agg")
ds_group = create_spark_dataset(
    pd.DataFrame({
        'group': ['A', 'A', 'B', 'B', 'A'],
        'value': [10, 20, 30, 40, 15]
    }),
    roles={'group': FeatureRole(), 'value': TargetRole(float)}
)
grouped = ds_group.groupby("group")
assert grouped is not None
grouped_agg = grouped.agg({"value": "mean"})
assert isinstance(grouped_agg, Dataset)
print("✅ groupby + agg — OK")

print("🔹 Тест: merge")
left = create_spark_dataset(
    pd.DataFrame({'key': [1, 2], 'val1': [10, 20]}), 
    roles={'key': InfoRole(), 'val1': TargetRole(int)}
)
right = create_spark_dataset(
    pd.DataFrame({'key': [1, 2], 'val2': [100, 200]}), 
    roles={'key': InfoRole(), 'val2': TargetRole(int)}
)
merged = left.merge(right, on="key")
assert "val1" in merged.columns and "val2" in merged.columns
print("✅ merge — OK")

🔹 Тест: groupby + agg
✅ groupby + agg — OK
🔹 Тест: merge
✅ merge — OK


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [9]:
print("🔹 Тест: dropna / fillna / isna")
df_na = pd.DataFrame({'a': [1, None, 3], 'b': [4, 5, None]})
ds_na = create_spark_dataset(df_na, roles={'a': InfoRole(), 'b': InfoRole()})
isna_result = ds_na.isna()
assert isinstance(isna_result, Dataset)
ds_filled = ds_na.fillna(values=0)
assert isinstance(ds_filled, Dataset)
ds_dropped = ds_na.dropna()
assert len(ds_dropped) < len(ds_na)
print("✅ dropna/fillna/isna — OK")

print("🔹 Тест: rename / replace / filter")
ds = create_spark_dataset()
renamed = ds.rename({"value": "new_value"})
assert isinstance(renamed, Dataset)
assert "new_value" in renamed.columns
replaced = ds.replace(to_replace={"name": {"A": "Alpha"}})
assert isinstance(replaced, Dataset)
filtered = ds.filter(items=["id", "score"])
assert set(filtered.columns) == {"id", "score"}
print("✅ rename/replace/filter — OK")

print("🔹 Тест: select_dtypes / isin / sample")
numeric_only = ds.select_dtypes(include=[np.number])
assert isinstance(numeric_only, Dataset)
assert "name" not in numeric_only.columns
isin_result = ds.isin([1, 3])
assert isinstance(isin_result, Dataset)
sampled = ds.sample(n=2, random_state=42)
assert isinstance(sampled, Dataset)
assert len(sampled) == 2
print("✅ select_dtypes/isin/sample — OK")

🔹 Тест: dropna / fillna / isna
✅ dropna/fillna/isna — OK
🔹 Тест: rename / replace / filter
✅ rename/replace/filter — OK
🔹 Тест: select_dtypes / isin / sample


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `ind

✅ select_dtypes/isin/sample — OK


In [10]:
ds

,id,name,value,active,score
0,1,A,10.5,True,100
1,2,B,20.3,False,200
2,3,C,15.7,True,150
3,4,D,30.1,False,300
4,5,E,25.9,True,250


In [11]:
5 * 0.72

3.5999999999999996

In [12]:
ds.sample(n=3, random_state=42)

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,id,name,value,active,score
0,3,C,15.7,True,150
1,2,B,20.3,False,200
2,1,A,10.5,True,100


In [13]:
print("🔹 Тест: log")
ds = create_spark_dataset()
log_ds = ds[["value"]].log()
assert log_ds is not None
print("✅ log — OK")

print("🔹 Тест: cov / corr")
num_ds = ds
cov_matrix = num_ds.cov()
assert cov_matrix is not None
corr_matrix = num_ds.corr()
assert corr_matrix is not None
print("✅ cov/corr — OK")

🔹 Тест: log
✅ log — OK
🔹 Тест: cov / corr
num cols = ['id', 'value', 'score']


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_numpy` loads all data into the driver's memory. It should only be used if the resulting NumPy ndarray is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: Pandas

✅ cov/corr — OK


In [14]:

print("🔹 Тест: dot")
ds = create_spark_dataset()

# Тест 1: Одна колонка, один элемент вектора
single_col = ds[["score"]]
vec1 = np.array([2])
dot_result1 = single_col.dot(vec1)
assert dot_result1 is not None
print(f"✅ dot (1 колонка, np.array): {dot_result1}")

# Тест 2: Несколько колонок, вектор соответствует колонкам
multi_col = ds[["id", "score"]]
vec2 = np.array([1, 2])
dot_result2 = multi_col.dot(vec2)
assert dot_result2 is not None
print(f"✅ dot (2 колонки, np.array): {dot_result2}")

# # Тест 3: Dot с другим Dataset (одна колонка)
# ds_other = create_spark_dataset()
# dot_result3 = ds[["score"]].dot(ds_other[["score"]])
# assert dot_result3 is not None
# print(f"✅ dot (Dataset, 1 колонка): {dot_result3}")

# # Тест 4: Dot с другим Dataset (несколько колонок)
# dot_result4 = ds[["id", "score"]].dot(ds_other[["id", "score"]])
# assert dot_result4 is not None
# print(f"✅ dot (Dataset, 2 колонки): {dot_result4}")

# # Тест 5: Проверка ошибки при несовместимых размерах
# try:
#     wrong_vec = np.array([1, 2, 3, 4, 5])  # 5 элементов для 1 колонки
#     ds[["score"]].dot(wrong_vec)
#     print("❌ Ожидалась ошибка")
# except ValueError as e:
#     print(f"✅ Ошибка при несовместимых размерах: {e}")

print("✅ dot — OK")

🔹 Тест: dot


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


✅ dot (1 колонка, np.array):    None
0   200
1   400
2   300
3   600
4   500
✅ dot (2 колонки, np.array):    None
0   201
1   402
2   303
3   604
4   505
✅ dot — OK


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [15]:
print("🔹 Тест: value_counts (Pandas vs Spark)")

# Pandas
df_pd = pd.DataFrame({'cat': ['X', 'Y', 'X', 'Z', 'X']})
ds_pd = Dataset({'cat': FeatureRole()}, data=df_pd, backend=BackendsEnum.pandas)
vc_pd = ds_pd.value_counts()
print(f"Pandas колонки: {vc_pd.columns}")  # ['cat', 'count']

# Spark
df_sp = pd.DataFrame({'cat': ['X', 'Y', 'X', 'Z', 'X']})
ds_sp = Dataset({'cat': FeatureRole()}, data=df_sp, backend=BackendsEnum.spark, session=spark)
vc_sp = ds_sp.value_counts()
print(f"Spark колонки: {vc_sp.columns}")  # Должно быть ['cat', 'count']

# assert vc_pd.columns == vc_sp.columns  # ✅ Теперь совпадают
# print("✅ value_counts — OK")

print("🔹 Тест: na_counts")
df_na = pd.DataFrame({'a': [1, None, 3], 'b': [None, None, 3]})
ds_na = create_spark_dataset(df_na, roles={'a': InfoRole(), 'b': InfoRole()})
na_counts = ds_na.na_counts()
assert na_counts is not None
print("✅ na_counts — OK")

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/base.py:1437: FutureWarning: The resulting Series will have a fixed name of 'count' from 4.0.0.
  warnings.warn(
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


🔹 Тест: value_counts (Pandas vs Spark)
Pandas колонки: Index(['cat', 'count'], dtype='object')
Spark колонки: ['cat', 'count']
🔹 Тест: na_counts
✅ na_counts — OK


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [16]:
print("🔹 Тест: apply")
ds = create_spark_dataset()
applied = ds[["score"]].apply(lambda x: x * 2, role={"score": StatisticRole()})
assert isinstance(applied, Dataset)
print("✅ apply — OK")

print("🔹 Тест: map")
mapped = ds[["id"]].map(lambda x: x + 10)
assert isinstance(mapped, Dataset)
print("✅ map — OK")

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


🔹 Тест: apply


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


✅ apply — OK
🔹 Тест: map
✅ map — OK


/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
